# 📡 Seeing Signals — Part II
## VLM Fine-Tuning: SmolVLM-256M-Instruct + LoRA

**Workflow:**
1. Configure local paths & install dependencies
2. Load dataset & prepare VLM training samples
3. Load SmolVLM-256M-Instruct
4. Apply LoRA (parameter-efficient fine-tuning)
5. Training loop
6. Evaluation & comparison with CNN

In [ ]:
from pathlib import Path

DATASET_DIR = Path('../data/generated')
CNN_DIR = Path('../results/cnn')
OUTPUT_DIR = Path('../results/vlm')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset directory: {DATASET_DIR.resolve()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')


## Cell 2 — Install Dependencies

In [ ]:
!pip install transformers peft accelerate bitsandbytes pandas pillow tqdm -q
!pip install torch torchvision -q

%matplotlib inline
import torch
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import json, os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 — Load Dataset & Prepare VLM Samples

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATASET_DIR / 'labels.csv')
print(f'Total samples: {len(df)}')

# Train/Val/Test split (same as CNN: 70/15/15)
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42,
                                      stratify=df['modulation'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42,
                                      stratify=temp_df['modulation'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# VLM question-answer pairs
# Each sample supports five question-answer tasks
QUESTIONS = {
    'modulation':   'What modulation scheme is used in this constellation?',
    'snr_range':    'What is the SNR range of this signal? Answer with: low, medium, or high.',
    'phase_noise':  'What is the phase noise severity? Answer with: none, mild, moderate, or severe.',
    'iq_imbalance': 'What is the I/Q imbalance severity? Answer with: none, mild, moderate, or severe.',
    'jamming':      'Is there jamming or external interference? Answer with: none or present.',
}

print('\nVLM Questions:')
for task, q in QUESTIONS.items():
    print(f'  [{task}] {q}')

## Cell 4 — Load SmolVLM-256M-Instruct

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded! Total parameters: {total_params/1e6:.1f}M')
print(f'Model dtype: {next(model.parameters()).dtype}')

## Cell 5 — Apply LoRA (Parameter-Efficient Fine-Tuning)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank
    lora_alpha=32,                 # scaling factor
    target_modules=['q_proj', 'v_proj'],  # apply to attention Q and V
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Trainable vs frozen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)')

## Cell 6 — VLM Dataset Class

In [ ]:
from torch.utils.data import Dataset, DataLoader

class VLMConstellationDataset(Dataset):
    """
    Each sample: (image, question) → answer
    Task selection is determined by the dataset index.
    """
    def __init__(self, df, img_dir, processor, tasks=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.processor = processor
        self.tasks     = tasks or list(QUESTIONS.keys())

    def __len__(self):
        return len(self.df) * len(self.tasks)

    def __getitem__(self, idx):
        row_idx  = idx // len(self.tasks)
        task_idx = idx %  len(self.tasks)
        row      = self.df.iloc[row_idx]
        task     = self.tasks[task_idx]

        img      = Image.open(self.img_dir / row['filename']).convert('RGB')
        question = QUESTIONS[task]
        answer   = str(row[task])

        # Format as chat
        messages = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': question},
                ]
            },
            {
                'role': 'assistant',
                'content': [{'type': 'text', 'text': answer}]
            }
        ]

        prompt = self.processor.apply_chat_template(messages, tokenize=False)
        inputs = self.processor(
            text=prompt,
            images=[img],
            return_tensors='pt',
            truncation=True,
            max_length=256,
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}


IMG_DIR = DATASET_DIR / 'images'

# Use a subset to keep memory requirements manageable
# Train across all five classification tasks
TRAIN_TASKS = ['modulation', 'snr_range', 'phase_noise', 'iq_imbalance', 'jamming']

train_ds = VLMConstellationDataset(train_df.sample(3000, random_state=42),
                                    IMG_DIR, processor, TRAIN_TASKS)
val_ds   = VLMConstellationDataset(val_df.sample(500, random_state=42),
                                    IMG_DIR, processor, TRAIN_TASKS)

print(f'VLM Train samples: {len(train_ds)}')
print(f'VLM Val samples:   {len(val_ds)}')

## Cell 7 — Training Loop

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def collate_fn(batch):
    keys = batch[0].keys()
    return {
        k: torch.nn.utils.rnn.pad_sequence(
            [b[k] for b in batch], batch_first=True, padding_value=0
        ) for k in keys
    }

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                           collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                           collate_fn=collate_fn, num_workers=2)

N_EPOCHS  = 3
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=N_EPOCHS * len(train_loader),
)

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
best_model_path = OUTPUT_DIR / 'best_vlm'

for epoch in range(N_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{N_EPOCHS} [Train]',
                       leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch, labels=batch['input_ids'])
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch, labels=batch['input_ids'])
            val_loss += outputs.loss.item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(best_model_path)
        processor.save_pretrained(best_model_path)
        print(f'  ✓ Best model saved')

print(f'\nTraining complete! Best Val Loss: {best_val_loss:.4f}')

## Cell 8 — Evaluation: Accuracy per Task

In [ ]:
from peft import PeftModel

# Load best model
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model_eval = PeftModel.from_pretrained(base_model, best_model_path)
model_eval.eval()
processor_eval = AutoProcessor.from_pretrained(best_model_path)

def predict_vlm(image, question, max_new_tokens=20):
    """Generate answer for a single image-question pair."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': question},
        ]
    }]
    prompt = processor_eval.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True)
    inputs = processor_eval(
        text=prompt, images=[image],
        return_tensors='pt'
    ).to(DEVICE)

    with torch.no_grad():
        output = model_eval.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return processor_eval.decode(generated, skip_special_tokens=True).strip().lower()


# Evaluate on a test subset for practical runtime
test_subset = test_df.sample(500, random_state=42)
results = {task: {'correct': 0, 'total': 0} for task in TRAIN_TASKS}

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc='Evaluating VLM'):
    img = Image.open(os.path.join(IMG_DIR, row['filename'])).convert('RGB')
    for task in TRAIN_TASKS:
        pred   = predict_vlm(img, QUESTIONS[task])
        true   = str(row[task]).lower()
        results[task]['correct'] += int(true in pred or pred in true)
        results[task]['total']   += 1

print('\n=== VLM Test Accuracy per Task ===')
vlm_accs = {}
for task, r in results.items():
    acc = r['correct'] / r['total'] * 100
    vlm_accs[task] = round(acc, 2)
    print(f'  {task:15s}: {acc:.2f}%')

## Cell 9 — CNN vs VLM Comparison Plot

In [ ]:
# Load CNN results
with open(CNN_DIR / 'cnn_results.json') as f:
    cnn_results = json.load(f)

tasks_plot = ['modulation', 'snr_range', 'phase_noise', 'iq_imbalance', 'jamming']
task_labels = ['Modulation', 'SNR Range', 'Phase Noise', 'IQ Imbalance', 'Jamming']

cnn_accs = [cnn_results.get(t, 0) for t in tasks_plot]
vlm_accs_list = [vlm_accs.get(t, 0) for t in tasks_plot]

x = np.arange(len(task_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, cnn_accs,     width, label='Custom CNN',    color='#3498db')
bars2 = ax.bar(x + width/2, vlm_accs_list, width, label='SmolVLM-256M', color='#e74c3c')

ax.set_xlabel('Task', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('CNN vs VLM — Classification Accuracy per Task', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(task_labels)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cnn_vs_vlm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison plot saved')

## Cell 10 — Save VLM Results

In [ ]:
vlm_summary = {
    'model': MODEL_ID,
    'lora_rank': 16,
    'lora_alpha': 32,
    'n_epochs': N_EPOCHS,
    'best_val_loss': round(best_val_loss, 4),
    'test_accuracy': vlm_accs,
    'trainable_params': trainable,
    'total_params': total,
}

with open(OUTPUT_DIR / 'vlm_results.json', 'w') as f:
    json.dump(vlm_summary, f, indent=2)

print('=== VLM Summary ===')
for k, v in vlm_summary.items():
    print(f'  {k}: {v}')

## Cell 11 — Qualitative Examples

In [ ]:
# Show qualitative prediction examples
examples = test_df.sample(6, random_state=99)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('VLM Qualitative Examples — Modulation Classification', fontsize=13)

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    img  = Image.open(os.path.join(IMG_DIR, row['filename'])).convert('RGB')
    pred = predict_vlm(img, QUESTIONS['modulation'])
    true = row['modulation'].lower()
    color = 'green' if true in pred or pred in true else 'red'
    ax.imshow(img)
    ax.set_title(f'True: {row["modulation"]}\nPred: {pred}',
                  color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vlm_examples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Qualitative examples saved')